# Librería

In [8]:
# Manipulacion de datos
import pandas as pd
import numpy as np
import datetime
pd.set_option('display.max_columns', 200)
import json
import os
import unicodedata

import geopandas as gpd
from shapely.geometry import Point

from pathlib import Path

# Paths

In [9]:
csv_path = Path('/Users/jaydymarchan/Desktop/causalidad/data/01_raw/datos_temperatura_minima/202207010000TMin.csv/')

In [10]:
df_temp = pd.read_csv(csv_path, encoding='latin-1')
# Normalizar el nombre de las columnas a minúsculas
df_temp.columns = df_temp.columns.str.strip().str.lower()

In [11]:
df_temp

,lon,lat,clave,edo,est,tmin
0,-102.79,21.81,MNLAG,AGS,Media Luna Ags.,15.3
1,-102.71,21.85,CALVILLO,AGS,Calvillo Ags. SMN*,15.4
2,-102.29,21.85,OBSAG,AGS,Observatorio de Aguascalientes Ags.,15.7
3,-102.31,21.90,AGSAG,AGS,Aguascalientes Ags.,16.1
4,-101.99,21.90,CNSAG,AGS,Los Conos Ags.,14.0
...,...,...,...,...,...,...
1368,-103.61,21.22,HITZC,ZAC,Huitzila Zac.,13.4
1369,-102.27,22.58,OCAZC,ZAC,Ojo Caliente Zac.,14.2
1370,-102.68,22.90,MMZC,ZAC,Aeropuerto Internacional de Zacatecas Zac.*,14.2
1371,-102.86,22.12,TAYAHUA,ZAC,Tayahua Zac. SMN*,16.0


# Funciones

In [12]:
# Definir la ruta de la carpeta donde están las lluvias
carpeta_lluvias = Path('/Users/jaydymarchan/Desktop/causalidad/data/01_raw/datos_temperatura_minima/')
# Diccionario para obtener el número de entidad (cve_ent)
diccionario_edos = {
    'AGS': 1, 'BC': 2, 'BCN': 2, 'BCS': 3, 'CAMP': 4, 'COAH': 5, 
    'COL': 6, 'CHIS': 7, 'CHIH': 8, 'CMX': 9, 'DF': 9, 'CDMX': 9, 'DGO': 10, 
    'GTO': 11, 'GRO': 12, 'HGO': 13, 'JAL': 14, 'MEX': 15, 'MICH': 16, 
    'MOR': 17, 'NAY': 18, 'NL': 19, 'OAX': 20, 'PUE': 21, 'QRO': 22, 
    'QROO': 23, 'ROO': 23, 'SLP': 24, 'SIN': 25, 'SON': 26, 'TAB': 27, 
    'TAM': 28, 'TAMPS': 28, 'TLAX': 29, 'VER': 30, 'YUC': 31, 'ZAC': 32
}

# NUEVO: Diccionario para obtener el nombre completo a partir de la cve_ent
diccionario_nombres_edos = {
    1: 'Aguascalientes', 2: 'Baja California', 3: 'Baja California Sur', 
    4: 'Campeche', 5: 'Coahuila', 6: 'Colima', 7: 'Chiapas', 
    8: 'Chihuahua', 9: 'Ciudad de México', 10: 'Durango', 
    11: 'Guanajuato', 12: 'Guerrero', 13: 'Hidalgo', 14: 'Jalisco', 
    15: 'Estado de México', 16: 'Michoacán', 17: 'Morelos', 18: 'Nayarit', 
    19: 'Nuevo León', 20: 'Oaxaca', 21: 'Puebla', 22: 'Querétaro', 
    23: 'Quintana Roo', 24: 'San Luis Potosí', 25: 'Sinaloa', 
    26: 'Sonora', 27: 'Tabasco', 28: 'Tamaulipas', 29: 'Tlaxcala', 
    30: 'Veracruz', 31: 'Yucatán', 32: 'Zacatecas'
}

dataframes_lluvias = []

# Iterar sobre todos los archivos CSV en la carpeta
for archivo in carpeta_lluvias.glob('*.csv'):
    # Leer el archivo actual
    df_temp = pd.read_csv(archivo, encoding='latin-1')
    
    # 1. Identificar el nombre de la última columna
    nombre_col_lluvia = df_temp.columns[-1]
    
    # Renombrar esa columna a 'lluvia_mm'
    df_temp.rename(columns={nombre_col_lluvia: 'temp_min'}, inplace=True)
    df_temp.columns = df_temp.columns.str.strip().str.lower()
    
    # 2. Limpiar la columna CLAVE
    df_temp['CLAVE'] = df_temp['clave'].astype(str).str.strip()
    print(df_temp.columns )
    print(df_temp.head(1))


    # 3. Extraer el nombre del municipio (lo que está antes de la primera coma)
    df_temp['municipio'] = df_temp['est'].str.split(',').str[0].str.strip()
    
    # 4. Convertir EDO a número usando el primer diccionario
    df_temp['EDO'] = df_temp['edo'].astype(str).str.strip().str.upper()
    df_temp['cve_ent'] = df_temp['edo'].map(diccionario_edos)
    
    # 5. NUEVO: Asignar el nombre completo de la entidad usando el segundo diccionario
    df_temp['nombre_entidad'] = df_temp['cve_ent'].map(diccionario_nombres_edos)
    
    # 6. Extracción de año y mes desde el nombre del archivo
    nombre_archivo = archivo.name
    df_temp['anio'] = pd.to_numeric(nombre_archivo[:4])
    df_temp['mes'] = pd.to_numeric(nombre_archivo[4:6])
    
    # Agregar el DataFrame procesado a la lista
    dataframes_lluvias.append(df_temp)

# Concatenar todos los DataFrames en uno solo maestro
df_lluvias_final = pd.concat(dataframes_lluvias, ignore_index=True)

# Mostrar el resultado final
display(df_lluvias_final.head())

Index(['lon', 'lat', 'clave', 'edo', 'est', 'temp_min', 'CLAVE'], dtype='str')
      lon    lat  clave  edo              est  temp_min  CLAVE
0 -102.79  21.81  MNLAG  AGS  Media Luna Ags.       3.5  MNLAG
Index(['lon', 'lat', 'clave', 'edo', 'est', 'temp_min', 'CLAVE'], dtype='str')
     lon    lat  clave  edo          est  temp_min  CLAVE
0 -89.68  20.65  ABAYC  YUC  Abalá, Yuc.      11.6  ABAYC
Index(['lon', 'lat', 'clave', 'edo', 'est', 'temp_min', 'CLAVE'], dtype='str')
      lon    lat  clave  edo             est  temp_min  CLAVE
0 -102.29  21.85  76571  AGS  AGUASCALIENTES      11.6  76571
Index(['lon', 'lat', 'clave', 'edo', 'est', 'temp_min', 'CLAVE'], dtype='str')
      lon   lat clave  edo                                               est  \
0 -102.32  21.7  MMAS  AGS  Aeropuerto Internacional de Aguascalientes Ags.*   

   temp_min CLAVE  
0      10.4  MMAS  
Index(['lon', 'lat', 'clave', 'edo', 'est', 'temp_min', 'CLAVE'], dtype='str')
      lon    lat  clave  edo          

,lon,lat,clave,edo,est,temp_min,CLAVE,municipio,EDO,cve_ent,nombre_entidad,anio,mes,tmin,unnamed: 6,unnamed: 7
0,-102.79,21.81,MNLAG,AGS,Media Luna Ags.,3.5,MNLAG,Media Luna Ags.,AGS,1.0,Aguascalientes,2022,3,NaN,NaN,NaN
1,-102.71,21.85,CALVILLO,AGS,Calvillo Ags. SMN*,5.2,CALVILLO,Calvillo Ags. SMN*,AGS,1.0,Aguascalientes,2022,3,NaN,NaN,NaN
2,-102.29,21.85,OBSAG,AGS,Observatorio de Aguascalientes Ags.,8.3,OBSAG,Observatorio de Aguascalientes Ags.,AGS,1.0,Aguascalientes,2022,3,NaN,NaN,NaN
3,-102.31,21.90,AGSAG,AGS,Aguascalientes Ags.,9.9,AGSAG,Aguascalientes Ags.,AGS,1.0,Aguascalientes,2022,3,NaN,NaN,NaN
4,-101.99,21.90,CNSAG,AGS,Los Conos Ags.,4.6,CNSAG,Los Conos Ags.,AGS,1.0,Aguascalientes,2022,3,NaN,NaN,NaN


# Sanity check: cobertura de meses y años (2013–2025)

In [13]:
# =========================================================
# SANITY CHECK: ¿están todos los meses y años 2013–2025?
# =========================================================
anio_ini, anio_fin = 2013, 2025

# --- 1) Cobertura segun los ARCHIVOS en la carpeta -------------------------
archivos = sorted(carpeta_lluvias.glob('*.csv'))
periodos_archivos = set()
for a in archivos:
    y, m = int(a.name[:4]), int(a.name[4:6])
    periodos_archivos.add((y, m))

# --- 2) Cobertura segun el DataFrame ya cargado ---------------------------
periodos_df = set(
    map(tuple, df_lluvias_final[['anio', 'mes']].drop_duplicates().to_numpy())
)

# --- 3) Rejilla esperada (13 años x 12 meses = 156 periodos) -------------
esperados = {(y, m) for y in range(anio_ini, anio_fin + 1) for m in range(1, 13)}

falt_archivos = sorted(esperados - periodos_archivos)
falt_df       = sorted(esperados - periodos_df)
extra_df      = sorted(periodos_df - esperados)          # periodos fuera de rango
solo_archivos = sorted(periodos_archivos - periodos_df)  # archivo existe pero no llegó al df

print(f"Periodos esperados      : {len(esperados)}  ({anio_ini}-01 a {anio_fin}-12)")
print(f"Periodos en archivos    : {len(periodos_archivos)}")
print(f"Periodos en df_lluvias  : {len(periodos_df)}")
print()
print(f"FALTAN en archivos ({len(falt_archivos)}):")
print("  " + (", ".join(f"{y}-{m:02d}" for y, m in falt_archivos) or "ninguno ✔"))
print()
print(f"FALTAN en df_lluvias_final ({len(falt_df)}):")
print("  " + (", ".join(f"{y}-{m:02d}" for y, m in falt_df) or "ninguno ✔"))
print()
print(f"Periodos FUERA de rango 2013–2025 en el df ({len(extra_df)}):")
print("  " + (", ".join(f"{y}-{m:02d}" for y, m in extra_df) or "ninguno ✔"))
print()
print(f"Archivos que NO aparecen en el df ({len(solo_archivos)}):")
print("  " + (", ".join(f"{y}-{m:02d}" for y, m in solo_archivos) or "ninguno ✔"))

# --- 4) Matriz visual año x mes (1 = presente en archivos, . = falta) ----
print("\nMatriz de cobertura (archivos)   1 = presente | . = falta")
print("año  " + " ".join(f"{m:02d}" for m in range(1, 13)))
for y in range(anio_ini, anio_fin + 1):
    fila = " ".join(" 1" if (y, m) in periodos_archivos else "  ." for m in range(1, 13))
    print(f"{y} {fila}")

# --- 5) Conteo de registros y estaciones por periodo ---------------------
resumen_periodos = (
    df_lluvias_final
    .groupby(['anio', 'mes'])
    .agg(n_registros=('temp_min', 'size'),
         n_estaciones=('CLAVE', 'nunique'),
         temp_min_nula=('temp_min', lambda s: s.isna().sum()))
    .reset_index()
    .sort_values(['anio', 'mes'])
)
display(resumen_periodos)

Periodos esperados      : 156  (2013-01 a 2025-12)
Periodos en archivos    : 132
Periodos en df_lluvias  : 132

FALTAN en archivos (24):
  2013-09, 2013-10, 2013-11, 2013-12, 2014-01, 2014-02, 2014-03, 2014-04, 2014-05, 2014-06, 2014-07, 2014-08, 2014-09, 2014-10, 2014-11, 2014-12, 2025-05, 2025-06, 2025-07, 2025-08, 2025-09, 2025-10, 2025-11, 2025-12

FALTAN en df_lluvias_final (24):
  2013-09, 2013-10, 2013-11, 2013-12, 2014-01, 2014-02, 2014-03, 2014-04, 2014-05, 2014-06, 2014-07, 2014-08, 2014-09, 2014-10, 2014-11, 2014-12, 2025-05, 2025-06, 2025-07, 2025-08, 2025-09, 2025-10, 2025-11, 2025-12

Periodos FUERA de rango 2013–2025 en el df (0):
  ninguno ✔

Archivos que NO aparecen en el df (0):
  ninguno ✔

Matriz de cobertura (archivos)   1 = presente | . = falta
año  01 02 03 04 05 06 07 08 09 10 11 12
2013  1  1  1  1  1  1  1  1   .   .   .   .
2014   .   .   .   .   .   .   .   .   .   .   .   .
2015  1  1  1  1  1  1  1  1  1  1  1  1
2016  1  1  1  1  1  1  1  1  1  1  1  1
20

,anio,mes,n_registros,n_estaciones,temp_min_nula
0,2013,1,939,939,0
1,2013,2,949,949,0
2,2013,3,886,886,0
3,2013,4,948,948,0
4,2013,5,946,946,0
...,...,...,...,...,...
127,2024,12,1233,1233,0
128,2025,1,1325,1325,0
129,2025,2,1383,1383,0
130,2025,3,1378,1378,0


In [14]:
# 1. Cargar el shapefile geográfico
ruta_shp = '/Users/jaydymarchan/Desktop/causalidad/data/01_raw/datos_lat_longitud/2024_1_00_MUN.shp'


# 1. Cargar el shapefile geográfico
# ruta_shp = '/Users/jaydymarchan/Desktop/causalidad/data/01_raw/datos_lat_longitud/2024_1_00_ENT.shp'
gdf_poligonos = gpd.read_file(ruta_shp)

# --- CORRECCIÓN: Si el shapefile no tiene CRS asignado, se lo definimos (INEGI suele usar WGS84 por defecto) ---
if gdf_poligonos.crs is None:
    gdf_poligonos.set_crs("EPSG:4326", inplace=True)

# Asegurar que el sistema de coordenadas final sea el estándar (Lat/Lon WGS84)
if gdf_poligonos.crs != "EPSG:4326":
    gdf_poligonos = gdf_poligonos.to_crs("EPSG:4326")

# 2. Convertir tu tabla de lluvias en un GeoDataFrame
# (Asegúrate de que df_lluvias_final tenga las columnas 'LON' y 'LAT' previamente limpias)
geometria_estaciones = [Point(xy) for xy in zip(df_lluvias_final['lon'], df_lluvias_final['lat'])]
gdf_estaciones = gpd.GeoDataFrame(df_lluvias_final, geometry=geometria_estaciones, crs="EPSG:4326")

# 3. Realizar el cruce espacial (Spatial Join)
gdf_cruce = gpd.sjoin(gdf_estaciones, gdf_poligonos, how="inner", predicate="within")

display(gdf_cruce.head())


,lon,lat,clave,edo,est,temp_min,CLAVE,municipio,EDO,cve_ent,nombre_entidad,anio,mes,tmin,unnamed: 6,unnamed: 7,geometry,index_right,CVEGEO,CVE_ENT,CVE_MUN,NOMGEO
0,-102.79,21.81,MNLAG,AGS,Media Luna Ags.,3.5,MNLAG,Media Luna Ags.,AGS,1.0,Aguascalientes,2022,3,NaN,NaN,NaN,POINT (-102.79 21.81),6,01003,01,003,Calvillo
1,-102.71,21.85,CALVILLO,AGS,Calvillo Ags. SMN*,5.2,CALVILLO,Calvillo Ags. SMN*,AGS,1.0,Aguascalientes,2022,3,NaN,NaN,NaN,POINT (-102.71 21.85),6,01003,01,003,Calvillo
2,-102.29,21.85,OBSAG,AGS,Observatorio de Aguascalientes Ags.,8.3,OBSAG,Observatorio de Aguascalientes Ags.,AGS,1.0,Aguascalientes,2022,3,NaN,NaN,NaN,POINT (-102.29 21.85),2,01001,01,001,Aguascalientes
3,-102.31,21.90,AGSAG,AGS,Aguascalientes Ags.,9.9,AGSAG,Aguascalientes Ags.,AGS,1.0,Aguascalientes,2022,3,NaN,NaN,NaN,POINT (-102.31 21.9),2,01001,01,001,Aguascalientes
4,-101.99,21.90,CNSAG,AGS,Los Conos Ags.,4.6,CNSAG,Los Conos Ags.,AGS,1.0,Aguascalientes,2022,3,NaN,NaN,NaN,POINT (-101.99 21.9),7,01010,01,010,El Llano


In [15]:
df_final = gdf_cruce[['CVE_ENT', 'CVE_MUN',	'nombre_entidad', 'NOMGEO', 'anio',	'mes', 'temp_min']]
df_lluvias_agrupadas = df_final.groupby(['CVE_ENT', 'CVE_MUN', 'nombre_entidad', 'NOMGEO','anio', 'mes']).agg({'temp_min': 'min'}).reset_index()
# ignorar la columna de cve_ent con el que viene los datos originales

In [16]:
# DataFrame base con temperatura mínima mensual
df_lluvias_agrupadas = df_final.groupby(['CVE_ENT', 'CVE_MUN', 'nombre_entidad', 'NOMGEO', 'anio', 'mes']).agg({'temp_min': 'min'}).reset_index()

# 1. Asegurar tipos de datos correctos
df_lluvias_agrupadas['anio'] = df_lluvias_agrupadas['anio'].astype(int)
df_lluvias_agrupadas['mes'] = df_lluvias_agrupadas['mes'].astype(int)

cols_base = ['CVE_ENT', 'CVE_MUN', 'nombre_entidad', 'NOMGEO']

# --- PASO EXTRA: Calcular la temperatura mínima absoluta de TODO el año anterior por municipio ---
# Obtenemos la temperatura más baja registrada en los 12 meses de cada año
tmin_anual_total = df_lluvias_agrupadas.groupby(
    cols_base + ['anio'], 
    as_index=False
)['temp_min'].agg(lambda x: x.min(skipna=True))

# Renombramos la columna para identificarla claramente
tmin_anual_total.rename(columns={'temp_min': 'temp_min_anual'}, inplace=True)

# Creamos una copia desplazando el año +1 para que el año anterior quede listo para unirse por 'anio' actual
tmin_anual_anterior = tmin_anual_total.copy()
tmin_anual_anterior['anio'] = tmin_anual_anterior['anio'] + 1
tmin_anual_anterior.rename(columns={'temp_min_anual': 'temp_min_anual_anterior'}, inplace=True)


# --- PROCESAMIENTO DE CICLOS (OI y PV) ---
resultados_ciclos = []
anos_unicos = range(2013, 2025)

for anio in anos_unicos:
    # Ciclo Otoño-Invierno (OI): Nov(t-1) a Abr(t)
    condicion_oi = (
        ((df_lluvias_agrupadas['anio'] == anio - 1) & (df_lluvias_agrupadas['mes'].isin([11, 12]))) |
        ((df_lluvias_agrupadas['anio'] == anio) & (df_lluvias_agrupadas['mes'].isin([1, 2, 3, 4])))
    )
    df_oi = df_lluvias_agrupadas[condicion_oi]
    
    # Para la temperatura del ciclo, usamos .min() en lugar de .sum()
    tmin_oi = df_oi.groupby(cols_base, as_index=False)['temp_min'].agg(lambda x: x.min(skipna=True))
    tmin_oi['anio'] = anio
    tmin_oi['nomcicloproductivo'] = 'OI'
    tmin_oi.rename(columns={'temp_min': 'temp_min_ciclo_min'}, inplace=True)
    
    # Ciclo Primavera-Verano (PV): Abr(t) a Sep(t)
    condicion_pv = (
        (df_lluvias_agrupadas['anio'] == anio) & (df_lluvias_agrupadas['mes'].isin([4, 5, 6, 7, 8, 9]))
    )
    df_pv = df_lluvias_agrupadas[condicion_pv]
    
    tmin_pv = df_pv.groupby(cols_base, as_index=False)['temp_min'].agg(lambda x: x.min(skipna=True))
    tmin_pv['anio'] = anio
    tmin_pv['nomcicloproductivo'] = 'PV'
    tmin_pv.rename(columns={'temp_min': 'temp_min_ciclo_min'}, inplace=True)
    
    resultados_ciclos.append(tmin_oi)
    resultados_ciclos.append(tmin_pv)

df_tmin_ciclos = pd.concat(resultados_ciclos, ignore_index=True)


# --- INTEGRAR LA TEMPERATURA MÍNIMA DEL AÑO ANTERIOR ---
# Hacemos un merge utilizando las llaves geográficas y el año
df_tmin_ciclos_con_lag = pd.merge(
    df_tmin_ciclos,
    tmin_anual_anterior,
    on=cols_base + ['anio'],
    how='left'
)

# Reordenar columnas para mayor legibilidad
df_tmin_ciclos_con_lag = df_tmin_ciclos_con_lag[
    cols_base + ['anio', 'nomcicloproductivo', 'temp_min_ciclo_min', 'temp_min_anual_anterior']
]

display(df_tmin_ciclos_con_lag.head(10))

,CVE_ENT,CVE_MUN,nombre_entidad,NOMGEO,anio,nomcicloproductivo,temp_min_ciclo_min,temp_min_anual_anterior
0,01,001,Aguascalientes,Aguascalientes,2013,OI,0.2,NaN
1,01,002,Aguascalientes,Asientos,2013,OI,2.6,NaN
2,01,003,Aguascalientes,Calvillo,2013,OI,1.4,NaN
3,01,004,Aguascalientes,Cosío,2013,OI,3.8,NaN
4,01,006,Aguascalientes,Pabellón de Arteaga,2013,OI,5.3,NaN
5,01,007,Aguascalientes,Rincón de Romos,2013,OI,5.7,NaN
6,01,008,Aguascalientes,San José de Gracia,2013,OI,1.8,NaN
7,01,009,Aguascalientes,Tepezalá,2013,OI,7.3,NaN
8,01,010,Aguascalientes,El Llano,2013,OI,1.8,NaN
9,02,001,Baja California,Ensenada,2013,OI,3.2,NaN


In [17]:
df_tmin_ciclos_con_lag.to_csv('/Users/jaydymarchan/Desktop/causalidad/data/02_processed/datos_temp_min.csv', index=False, encoding='utf-8')